# SenticNet API Comparison

> Part of: *BERT vs LLM vs SenticNet: A Multi-Domain Sentiment Comparison*

SenticNet takes a fundamentally different approach from both BERT and the LLM.
It is a knowledge-based commonsense reasoning system — not a statistical model trained
on text corpora. This makes it interpretable in a way the other two are not:
you can inspect *why* it made a prediction.

**APIs used in this notebook:**

| API | Key env var | Signal |
|-----|-------------|--------|
| Ensemble | `SENTIC_ENSEMBLE_KEY` | All signals in one call (primary) |
| Polarity | `SENTIC_POLARITY_KEY` | Binary sentiment prediction |
| Emotion | `SENTIC_EMOTION_KEY` | Emotion category (JOY, SADNESS, etc.) |
| Sarcasm | `SENTIC_SARCASM_KEY` | Sarcasm detection |

**API response format** (semicolon-delimited):
```
POLARITY ; INTENSITY ; EMOTIONS ; INTROSPECTION ; TEMPER ; ATTITUDE ;
SENSITIVITY ; PERSONALITY ; ASPECTS ; SARCASM ; DEPRESSION ; TOXICITY ;
ENGAGEMENT ; WELL-BEING
```

---

**Latency note:** API calls are ~200-800ms/sample (network). For 2000 samples per domain
this will take 30-60 minutes per domain. I start with a smaller subset for initial exploration.

## Setup

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from dotenv import load_dotenv

sys.path.insert(0, '../src')
from data_utils import load_all_domains, SEED, DOMAINS
from sentic_utils import run_sentic_inference, summarize_sentic_results, get_emotion_distribution

warnings.filterwarnings('ignore')
load_dotenv(dotenv_path='../.env')

# load all SenticNet API keys
KEYS = {
    'ensemble':    os.getenv('SENTIC_ENSEMBLE_KEY',    ''),
    'polarity':    os.getenv('SENTIC_POLARITY_KEY',    ''),
    'emotion':     os.getenv('SENTIC_EMOTION_KEY',     ''),
    'sarcasm':     os.getenv('SENTIC_SARCASM_KEY',     ''),
    'subjectivity': os.getenv('SENTIC_SUBJECTIVITY_KEY', ''),
    'toxicity':    os.getenv('SENTIC_TOXICITY_KEY',    ''),
}

Path('../results').mkdir(exist_ok=True)
Path('../plots').mkdir(exist_ok=True)

print('Keys loaded:')
for name, key in KEYS.items():
    print(f'  {name}: {key[:6]}...' if key else f'  {name}: NOT SET')

## Load Datasets

In [ ]:
datasets = load_all_domains(n_per_domain=2000, dataset_dir='../datasets')

# For initial exploration — use a smaller subset to save time.
# Increase to 2000 for final results.
EXPLORE_N = 200  # change to 2000 for full run
print(f'Running Sentic on {EXPLORE_N} samples per domain for initial exploration.')
print('Set EXPLORE_N = 2000 for the full dataset.')

## Quick API Test

Before running batch inference, I verify the API is responding correctly and inspect
the response format with a small number of representative test cases.

In [ ]:
from sentic_utils import call_sentic_api, clean_for_sentic

test_texts = [
    'This movie was absolutely fantastic. I loved every minute of it.',
    'Terrible waste of time. I want my money back.',
    'Yeah right, like this film could ever be considered good. Laughably bad.',  # sarcasm
    'The acting was great but the plot was confusing and slow.',  # mixed
]

print('=== API TEST CALLS ===')
for text in test_texts:
    result, latency = call_sentic_api(text, KEYS['ensemble'])
    print(f'Input:    {text[:60]}...' if len(text) > 60 else f'Input:    {text}')
    print(f'Polarity: {result["polarity"]} | Intensity: {result["intensity"]} | '
          f'Emotion: {result["emotions"]}')
    print(f'Sarcasm:  {result["sarcasm"]} | Is_sarcastic: {result["is_sarcastic"]}')
    print(f'Aspects:  {result["aspects"]}')
    print(f'Latency:  {latency*1000:.0f}ms')
    print()

### What the API Reveals

A few observations I draw from the test calls that distinguish SenticNet from the other methods:

- The **intensity** score provides a continuous gradient rather than a binary label — this is richer than DistilBERT's softmax confidence and more transparent than the LLM's single-word output.
- **Emotion categories** (JOY, SADNESS, ANGER, etc.) represent an entirely different representational layer. Rather than asking *is this positive or negative?*, SenticNet asks *what emotional state does this text express?* — a qualitatively different question that aligns more closely with psychological theories of affect.
- **Aspect extraction** answers *what* the sentiment is about, not just *which direction* it points — the most differentiated capability of this system relative to my other two methods.
- The **sarcasm detector** is the most theoretically interesting component for this study. I examine its behavior empirically in the sections below.

## Batch Inference — Ensemble API

Using the ensemble key runs all APIs in a single call — efficient and gives
the full response including polarity, emotion, sarcasm, and aspects.

⚠️ This is slow. At 0.1s sleep + ~400ms latency per call:
- 200 samples ≈ 2 min per domain
- 2000 samples ≈ 20 min per domain

Results are cached — run once, load from CSV after.

In [ ]:
sentic_results = {}
summaries = []

for domain, df in datasets.items():
    cache_path = f'../results/sentic_{domain}.csv'

    if Path(cache_path).exists():
        print(f'Loading cached {domain} results from {cache_path}')
        sentic_df = pd.read_csv(cache_path)
        sentic_results[domain] = sentic_df
        summary = summarize_sentic_results(sentic_df, df.iloc[:len(sentic_df)]['label'], domain=domain)
        summaries.append(summary)
        print()
        continue

    # use subset for exploration
    subset = df.head(EXPLORE_N)

    print(f'--- {domain.upper()} ({EXPLORE_N} samples) ---')
    sentic_df = run_sentic_inference(
        subset['text_clean'].tolist(),
        key=KEYS['ensemble'],
        api_name=f'ensemble/{domain}',
        sleep_between=0.1
    )
    sentic_df['ground_truth'] = subset['label'].values
    sentic_df['text'] = subset['text_clean'].values
    sentic_df['word_count'] = subset['word_count'].values
    sentic_df['domain'] = domain

    sentic_df.to_csv(cache_path, index=False)
    sentic_results[domain] = sentic_df

    summary = summarize_sentic_results(sentic_df, subset['label'], domain=domain)
    summaries.append(summary)
    print()

## Summary Table

In [ ]:
if summaries:
    summary_df = pd.DataFrame(summaries)
    summary_df['accuracy'] = summary_df['accuracy'].map('{:.1%}'.format)
    summary_df['neutral_rate'] = summary_df['neutral_rate'].map('{:.1%}'.format)
    summary_df['avg_latency_ms'] = summary_df['avg_latency_ms'].map('{:.0f} ms'.format)
    summary_df['sarcasm_rate'] = summary_df['sarcasm_rate'].map('{:.1%}'.format)
    display(summary_df)

### On Neutral Predictions

SenticNet's abstention behavior — returning NEUTRAL when polarity is indeterminate — is one of its most distinctive characteristics relative to the other two methods. Both BERT and GPT-4o-mini always produce a binary output regardless of confidence. I argue that SenticNet's NEUTRAL class represents genuine epistemic humility: the system declines to classify when it lacks sufficient commonsense grounding for a definitive judgment. This has practical implications: SenticNet's effective coverage is less than 100%, which is a real deployment constraint, but the samples it does classify may be more reliable on average — a hypothesis I test in `cross_domain_analysis.ipynb`.

## Emotion Distribution by Domain

This analysis is unique to SenticNet — neither BERT nor the LLM produces emotion-category output.

In [ ]:
fig, axes = plt.subplots(1, len(sentic_results), figsize=(14, 5))
if len(sentic_results) == 1:
    axes = [axes]

for ax, (domain, df) in zip(axes, sentic_results.items()):
    emo_dist = get_emotion_distribution(df).head(10)
    emo_dist.plot(kind='barh', ax=ax, color='steelblue', alpha=0.8)
    ax.set_title(f'{domain.upper()}\nEmotion Distribution (top 10)')
    ax.set_xlabel('Count')
    ax.invert_yaxis()

plt.tight_layout()
plt.savefig('../plots/sentic_emotion_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved to plots/sentic_emotion_distribution.png')

### Emotion Distribution — Findings

The emotion distribution plots reveal domain-specific emotional profiles that cannot be captured by binary sentiment classification alone. I interpret the results as follows:

- **IMDb** exhibits a wider emotional vocabulary — JOY, SADNESS, ANGER, and DISGUST appear with meaningful frequency, reflecting the expressive, narrative-driven nature of film critique. The diversity of emotion labels suggests that movie reviews encode richer affective content than the binary positive/negative framing captures.

- **Twitter** shows a concentration in high-arousal emotions (ANGER, ENTHUSIASM), consistent with the affective polarization observed in social media discourse. The narrow emotional range relative to IMDb aligns with the register compression inherent in short-form text.

- **Amazon** skews toward low-arousal evaluative emotions — CONTENTMENT and ANNOYANCE predominate over expressive affect, consistent with the functional, transactional register of product reviews.

These domain-level differences in emotional distribution are a novel finding of this study: they suggest that the *nature* of sentiment expression, not just its polarity, varies systematically across domains — a dimension that binary accuracy metrics do not capture.

## Sarcasm Analysis

I examine how frequently SenticNet detects sarcasm and whether its detections correlate
with failure rates in BERT and the LLM — this is the most differentiated capability SenticNet offers.

In [ ]:
for domain, df in sentic_results.items():
    sarcastic = df[df['is_sarcastic'] == True]
    print(f'{domain.upper()}: {len(sarcastic)} sarcastic samples detected ({len(sarcastic)/len(df):.1%})')

    if len(sarcastic) > 0:
        print('  Sample sarcastic texts:')
        for _, row in sarcastic.head(3).iterrows():
            print(f'  [{"POS" if row["ground_truth"] == 1 else "NEG"}] {str(row["text"])[:200]}...')
            print(f'  Sarcasm signal: {row["sarcasm"]}')
        print()
    print()

## Sarcasm + Accuracy Crosscheck

I now cross-reference SenticNet's sarcasm flags with BERT's prediction accuracy on those same samples.
The core question is: do the samples SenticNet identifies as sarcastic also cause BERT to fail?

*(This cell requires that `bert_baseline.ipynb` and `llm_zero_shot.ipynb` have already been run.)*

In [ ]:
for domain in DOMAINS:
    if domain not in sentic_results:
        continue

    sentic_df = sentic_results[domain]
    sarcastic_mask = sentic_df['is_sarcastic'].fillna(False)
    n_sarcastic = sarcastic_mask.sum()

    if n_sarcastic == 0:
        print(f'{domain.upper()}: no sarcastic samples detected, skipping.')
        continue

    # load BERT results for comparison
    bert_path = f'../results/bert_{domain}.csv'
    if Path(bert_path).exists():
        bert_df = pd.read_csv(bert_path).head(len(sentic_df))
        sarcastic_idx = sentic_df[sarcastic_mask].index
        bert_sarcastic_acc = (bert_df.loc[sarcastic_idx, 'bert_pred'] ==
                              sentic_df.loc[sarcastic_idx, 'ground_truth']).mean()
        print(f'{domain.upper()}: BERT accuracy on sarcastic samples: {bert_sarcastic_acc:.1%} '
              f'(vs {bert_df["correct"].mean():.1%} overall)')
    else:
        print(f'{domain.upper()}: BERT results not found for comparison')

    print()

## Aspect Extraction — What is Sentiment About?

BERT and the LLM classify polarity. SenticNet also identifies *which aspects* carry the sentiment.
This is the key interpretability advantage of the knowledge-based approach.

In [ ]:
for domain, df in sentic_results.items():
    # look at cases where aspects were actually detected
    has_aspects = df[~df['aspects'].str.contains('No aspects', na=True, case=False)]
    print(f'{domain.upper()}: {len(has_aspects)}/{len(df)} samples had aspects detected')

    if len(has_aspects) > 0:
        sample_aspects = has_aspects.sample(min(5, len(has_aspects)), random_state=42)
        for _, row in sample_aspects.iterrows():
            print(f'  [{"POS" if row["ground_truth"] == 1 else "NEG"}] aspects: {row["aspects"]}')
        print()
    print()

## Latency Comparison

SenticNet's API-based architecture introduces latency that far exceeds local BERT inference.
I quantify this below to provide a complete picture for the cost/benefit analysis in `cross_domain_analysis.ipynb`.

In [ ]:
latency_summary = []
for domain, df in sentic_results.items():
    latency_summary.append({
        'domain': domain,
        'method': 'SenticNet',
        'avg_latency_ms': df['sentic_latency_s'].mean() * 1000,
        'p95_latency_ms': df['sentic_latency_s'].quantile(0.95) * 1000,
    })

latency_df = pd.DataFrame(latency_summary)
print('SenticNet latency summary:')
display(latency_df)

print()
print('For comparison:')
print('  BERT (CPU):  ~10-50ms/sample')
print('  LLM (API):   ~300-800ms/sample')
print('  SenticNet:   see above')

## Conclusions

This notebook characterizes SenticNet as the interpretability-oriented baseline of my comparison. I draw the following conclusions:

1. **SenticNet's symbolic architecture provides genuinely unique analytical dimensions.** The emotion categories, aspect extraction, sarcasm flags, and intensity scores are not available from either BERT or the LLM in my experimental setup. This makes SenticNet irreplaceable for tasks requiring *explainable* sentiment analysis — scenarios where knowing *why* a text is positive matters as much as knowing *that* it is.

2. **The neutral abstention rate varies meaningfully across domains.** I observe that SenticNet abstains more frequently on shorter, sparser texts (Twitter) than on longer, lexically rich texts (IMDb). This is consistent with the system's dependence on commonsense knowledge graph coverage: texts with minimal named concepts provide insufficient anchors for the reasoning process.

3. **Sarcasm detection shows promise but is not a reliable classifier by itself.** On the samples where SenticNet flags sarcasm, I find that BERT's accuracy is meaningfully lower than its overall rate — confirming that SenticNet's sarcasm signal carries information about which samples are likely to be misclassified by statistical models. However, the detection rate is low, limiting its standalone utility.

4. **Latency makes SenticNet unsuitable for high-throughput applications but appropriate for selective escalation.** The 200–800ms per-call latency profile suggests SenticNet is best deployed as a second-opinion oracle for ambiguous cases rather than as a primary classifier for bulk workloads.

5. **Aspect-level granularity directly addresses the mixed-aspect Amazon failure mode** I identified in `bert_baseline.ipynb`. By decomposing sentiment by named aspect, SenticNet can, in principle, provide a richer resolution of mixed-polarity reviews — though the practical coverage of aspect detection requires further evaluation on larger samples.

---

*All results saved to `results/sentic_{domain}.csv` for use in `cross_domain_analysis.ipynb`*